<img src="./ccsf.png" alt="CCSF Logo" width=200px style="margin:0px -5px">

# Lab 06: Sampling and Testing Hypotheses

## References

* [Sections 10.0 - 10.4 of the Textbook](https://ccsf-math-108.github.io/textbook/chapters/10/Sampling_and_Empirical_Distributions.html)
* [Sections 11.0 - 11.2 of the Textbook](https://ccsf-math-108.github.io/textbook/chapters/11/Testing_Hypotheses.html)
* [datascience Documentation](https://datascience.readthedocs.io/)

---

## Lab Assignment Reminders

- 🚨 Make sure to run the code cell at the top of this notebook that starts with `# Initialize Otter` to load the auto-grader.
- Your tasks are categorized as auto-graded (📍) and manually graded (📍🔎):
    - **For all auto-graded tasks:**
        - Replace the `...` in the provided code cell with your own code.
        - Run the `grader.check` code cell to execute tests on your code.
        - There are no hidden auto-grader tests in the lab assignments. This means if you pass the tests, you can assume you've completed the task successfully.
    - **For all manually graded tasks:**
        - You may need to provide your own response to the provided prompt. Replace the template text "_Type your answer here, replacing this text._" with your own words.
        - You might need to produce a graphic or another output using code. Replace the `...` in the code cell to generate the image, table, etc.
        - In either case, check your response with a classmate, a tutor, or the instructor before moving on.
- Throughout this assignment and all future ones, please **do not re-assign variables** throughout the notebook! _For example, if you use `max_temperature` in your answer to one question, do not reassign it later on. Otherwise, you may fail tests that you thought you were passing previously!_
- You may [submit](#Submit-Your-Assignment-to-Canvas) this assignment as many times as you want before the deadline. Your instructor will score the last version you submit once the deadline has passed.
- **Collaborating on labs is encouraged!** You should rarely remain stuck for more than a few minutes on questions in labs, so ask an instructor or classmate for help. (Explaining things is beneficial, too -- the best way to solidify your knowledge of a subject is to explain it.) However, please don't just share answers.

---

## Set Up Notebook

### Task 00 📍

<!-- BEGIN QUESTION -->

Run the following code cells to set up the notebook.

_You are not responsible for understanding what this code does._

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook('lab06.ipynb')

In [ ]:
# Configure this Notebook
import numpy as np
from datascience import *
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

def sample_proportions(sample_size, probabilities):
    """Return the proportion of random draws for each outcome in a distribution.

    This function is similar to np.random.multinomial, but returns proportions
    instead of counts.

    Args:
        ``sample_size``: The size of the sample to draw from the distribution.

        ``probabilities``: An array of probabilities that forms a distribution.

    Returns:
        An array with the same length as ``probability`` that sums to 1.
    """
    return np.random.multinomial(sample_size, probabilities) / sample_size

<!-- END QUESTION -->

---

## Distributions

You recently learned about probability and empirical distributions. As a reminder:

* **Probability distributions** describe the theoretical likelihood of different outcomes in a random process. They are based on mathematical models and assumptions.
* **Empirical distributions** are derived from actual observed data. Instead of being based on theoretical probabilities, they reflect the frequencies of observed occurrences in a repeated experiment.

---

### Sampling Basketball Data

We will now move on to the topic of sampling which we discussed in more depth in this week's lectures. We'll guide you through this code, but if you wish to read more about different kinds of samples before attempting these tasks, you can check out [section 10 of the textbook](https://ccsf-math-108.github.io/textbook/chapters/10/Sampling_and_Empirical_Distributions.html).

We will be sampling from a data set that contains information about NBA players, including their ages and salaries. Run the cell below to load the data.

Run the cell below to load player and salary data that we will use for our sampling. 

In [ ]:
player_data = Table().read_table("player_data.csv")
salary_data = Table().read_table("salary_data.csv")
full_data = salary_data.join("PlayerName", player_data, "Name")

# The show method immediately displays the contents of a table. 
# This way, we can display the top of two tables using a single cell.
player_data.show(3)
salary_data.show(3)
full_data.show(3)

Rather than getting data on every player (as in the tables loaded above), imagine that we had gotten data on only a smaller subset of the players. For 492 players, it's not so unreasonable to expect to see all the data, but usually we aren't so lucky. 

If we want to make estimates about a certain numerical property of the population (known as a statistic, e.g. the mean or median), we may have to come up with these estimates based only on a smaller sample. Whether these estimates are useful or not often depends on how the sample was gathered. We have prepared some example sample datasets to see how they compare to the full NBA dataset. Later we'll ask you to create your own samples to see how they behave.

To save typing and increase the clarity of your code, we will package the analysis code into a few functions. This will be useful in the rest of the lab as we will repeatedly need to create histograms and collect summary statistics from that data.

We've defined the `histograms` function below, which takes a table with columns `Age` and `Salary` and draws a histogram for each one. It uses bin widths of 1 year for `Age` and $1,000,000 for `Salary`.

In [ ]:
def histograms(t):
    ages = t.column('Age')
    salaries = t.column('Salary')/1000000
    t1 = t.drop('Salary').with_column('Salary', salaries)
    age_bins = np.arange(min(ages), max(ages) + 2, 1) 
    salary_bins = np.arange(min(salaries), max(salaries) + 1, 1)
    t1.hist('Age', bins=age_bins, unit='year')
    plt.title('Age distribution')
    t1.hist('Salary', bins=salary_bins, unit='million dollars')
    plt.title('Salary distribution') 
    
histograms(full_data)
print('Two histograms should be displayed below')

#### Task 01 📍

Create a function called `compute_statistics` that takes a table containing ages and salaries and:
- Draws a histogram of ages
- Draws a histogram of salaries
- Returns a two-element array containing the average age and average salary (in that order)

You can call the `histograms` function to draw the histograms! 

*Note:* More charts will be displayed when running the test cell. Please feel free to ignore the charts.


In [ ]:
def compute_statistics(age_and_salary_data):
    ...
    age = ...
    salary = ...
    ...
    

full_stats = compute_statistics(full_data)
full_stats

In [ ]:
grader.check("task_01")

### Simple random sampling

One way to create a sample from a population is to sample uniformly at random from the population. In this demonstration, we are thinking of the population as the 492 players in the original data set. In a **simple random sample (SRS) without replacement**, we ensure that each player is selected at most once. Imagine writing down each player's name on a card, putting the cards in an box, and shuffling the box.  Then, pull out cards one by one and set them aside, stopping when the specified sample size is reached.



### Producing simple random samples

Sometimes, it’s useful to take random samples even when we have the data for the whole population. It helps us understand sampling accuracy.

### `sample`

The table method `sample` produces a random sample from the table. By default, it draws at random **with replacement** from the rows of a table. It takes in the sample size as its argument and returns a **table** with only the rows that were selected. 

Run the cell below to see an example call to `sample()` with a sample size of 5, with replacement. Because this is done with replacement, it is possible to see a player more than once in the resulting table.

In [ ]:
# Just run this cell

salary_data.sample(5)

The optional argument `with_replacement=False` can be passed through `sample()` to specify that the sample should be drawn without replacement.

Run the cell below to see an example call to `sample()` with a sample size of 5, without replacement. In this case, it is not possible to see a player more than once in the resulting table as it is done without replacement.

In [ ]:
# Just run this cell

salary_data.sample(5, with_replacement=False)

#### Task 02 📍🔎

<!-- BEGIN QUESTION -->

Produce a simple random sample of size 44 from `full_data`. Run your analysis (compute_statistics) on it again. Run the cell several times to see how the histograms and statistics change across different samples. **Then answer the questions below**:

- How much does the average age change across samples? 
- What about the average salary?

**Note:** Since this task does not have an auto-grader, make sure to check your results with a classmate, a tutor, or the instructor before moving on.

_Type your answer here, replacing this text._

In [ ]:
my_small_srswor_data = ...
my_small_stats = ...
my_small_stats

<!-- END QUESTION -->

#### Task 03 📍🔎

<!-- BEGIN QUESTION -->

As in the previous question, analyze several simple random samples of size 100 from full_data. **Then, answer the following questions**:
- Do the histogram shapes seem to change more or less across samples of 100 than across samples of size 44?  
- Are the sample averages and histograms closer to their true values/shape for age or for salary?  What did you expect to see?

**Note:** Since this task does not have an auto-grader, make sure to check your results with a classmate, a tutor, or the instructor before moving on.

_Type your answer here, replacing this text._

In [ ]:
my_large_srswor_data = ...
my_large_stats = ...
my_large_stats

<!-- END QUESTION -->

## Sampling from a Distribution

Suppose that you are studying a categorical variable from some population and you have a summary of the proportion of values. For example, some sandwich shop has a record from last year that 70% of their customers purchased chips with their sandwich and 30% did not. In this hypothetical situation, the variable `bought_chips` has two values `True` and `False`.

You could try to gather the original data or create a table that shows `True` on 70% of the rows and `False` on 30%. Instead, the `datascience` library has a function called `sample_proportions` that is helpful for generating random samples from a population like this. The basic format of the command is:

``` python 
sample_proportions(sample_size, probabilities)
```
where `sample_size` is the size of the sample and `probabilities` is an array of probabilities that reflect the chance of randomly selecting one of the variable's values.

If you wanted to simulate randomly sampling 3 customers from the sandwich shop customer population, you could define `sample_size = 3` and set up an array `population_arr = make_array(0.70, 0.30)` where the first value represents the chance of someone from the population purchasing chips with their sandwich. From that point running `sample_proportions(sample_size, population_arr)` would create an array of 2 values. The first value in the array would be the proportion `True` values (to represent the proportion of the 3 random customers that purchased chips) and the second value would be the proportion of `False` values (reflecting the customers that didn't purchase chips).

Run the following command to see how this works. You will get a variety of results, but you should see arrays with `1`, `0`, `0.66666667`, and `0.33333333` values.

In [ ]:
sample_size = 3
population_arr = make_array(0.70, 0.30)
sample_proportions(sample_size, population_arr)

### Task 04 📍

Suppose that a typical lunch rush for the sandwich shop consists of 30 sandwich purchases. Use the `sample_proportions` function to simulate randomly selecting 30 customers from a population where it is assumed that 70% purchase chips with their sandwich and 30% do not. Run this simulation 10,000 times and create an array called `chips` that contains the proportion of the randomly sampled 30 customers that bought chips. Finally, make a histogram of the proportions.

Your histogram should look **similar** to the following one where the histogram is approximately centered on the population value of 70% (0.7).

<img src="proportion_bought_chips.png" alt="histogram of 10,000 random sample" width = 40%>

In [ ]:
sample_size = ...
population_arr = ...

chips = make_array()

for ...:
    random_sample = ...
    prop_bought_chips = ...
    chips = np.append(chips, prop_bought_chips)

Table().with_column('bought_chips', ...).hist('bought_chips')
plt.title('Proportion Bought Chips (Sample Size=30)')
plt.show()

In [ ]:
grader.check("task_04")

---

## Therapeutic Touch

[Therapeutic touch (TT)](https://en.wikipedia.org/wiki/Therapeutic_touch) is an alternative medicine practice that is based on the idea that the human body has an energy field (called the human energy field, or HEF) that can be sensed, manipulated and balanced to promote health and relaxation. Practitioners of TT lightly place their hands on or above a patient's body to sense the patient's energy and then send healthy energy back.

---

### Emily Rosa

In 1996, [Emily Rosa](https://en.wikipedia.org/wiki/Emily_Rosa) was a 4th grade student who was very familiar with the world of TT, thanks to her parents, who were both medical practitioners and skeptics of TT. For her 4th grade science fair project, she designed an experiment that tested whether or not TT practitioners could truly sense a person's HEF. Her work was published in the Journal of the American Medical Association in 1998 making her the youngest person to have research published in a peer reviewed medical journal.

---

### Emily's Experiment

Emily's experiment was clean, simple, and effective. Due to her parents' occupations in the medical field, she had wide access to people who were TT practitioners.

Emily's experiment involved 21 TT practitioners and they were each given 10 trials. Each trial involved the practitioner extending both of their hands through a screen which they couldn't see through. On the other side of the screen, Emily placed her own hand near one of the practitioner's hands - either right or left based on the flip of a fair coin. She then asked the practitioner to tell her which of their hands detected her HEF. The idea is that if a practitioner can truly interact with a person's HEF, they should be able to answer this question correctly.

Overall, through the 210 trials, the practitioner picked the correct hand 44% of the time.

Emily's main goal here was to test whether or not the TT practitioners' guesses were random, like the flip of a coin. In many medical experiments, this is the norm. We test whether or not the treatment has an effect, not necessarily whether or not the treatment actually works.

We will now begin to formulate this experiment in terms of the terminology we learned in this course.

#### Task 05 📍

---

Emily suspected that the practitioners could not sense a person's HEF and were merely guessing at the correct hand. Using this idea, choose the option below that best describes Emily's model for how likely the TT practitioners are to choose the correct hand, and the alternative model hers is meant to discredit. Assign your choice to `models`.

1. Emily's model is that the TT practitioners have a 50% chance of choosing the correct hand. She is trying to debunk the alternative model that the TT practitioners have some chance other than 50% of choosing the correct hand. 
2. Emily's model is that the TT practitioners have have some chance other than 50% of choosing the correct hand. She is trying to debunk the alternative model that the TT practitioners have a 50% chance of choosing the correct hand. 

In [ ]:
models = ...

In [ ]:
grader.check("task_05")

---

#### Task 06 📍

Remember that the practitioner got the correct answer 44% (0.44) of the time. According to Emily's model, on average, what proportion of times do we expect the practitioner to guess the correct hand? Make sure your answer is between 0 and 1. 


In [ ]:
expected_proportion_correct = ...
expected_proportion_correct

In [ ]:
grader.check("task_06")

The goal now is to see if the deviation in the experimental results from this expected proportion is due to something other than chance.

---

#### Task 07 📍

We usually use a statistic to help determine which model the evidence points towards. What is a statistic that we can use to compare outcomes under Emily’s model to what was observed? Assign `valid_stat` to an array of integer(s) representing test statistics that Emily can use: 

1. The difference between the expected percent correct and the actual percent correct
2. The absolute difference between the expected percent correct and the actual percent correct
3. The sum of the expected percent correct and the actual percent correct



In [ ]:
valid_stat = ...
valid_stat

In [ ]:
grader.check("task_07")

---

#### Task 08 📍

Define the function `statistic` which takes in an expected proportion and an actual proportion, and returns the value of the statistic chosen in the previous task. Assume that the argument takes in proportions, but  return your answer as a percentage. 

*Hint:* Remember we are asking for a **percentage**, not a proportion. 


In [ ]:
def statistic(expected_prop, actual_prop):
    ...

In [ ]:
grader.check("task_08")

---

#### Task 09 📍

Use your newly defined function to calculate the observed statistic from Emily's experiment. 


In [ ]:
observed_statistic = ...
observed_statistic

In [ ]:
grader.check("task_09")

> **Is this observed statistic consistent with what we might see under Emily’s model?**

In order to answer this question, we must simulate the experiment as though Emily's model was correct, and calculate our statistic for every simulation.

---

### `sample_proportions`

`sample_proportions` can be used to randomly sample from multiple categories when you know the proportion of data points that are expected to fall in each category. `sample_proportions` takes two arguments: the sample size and an array that contains the distribution of categories in the population (should sum to 1).

Consider flipping a fair coin, where the two outcomes (coin lands heads and coin lands tails) occur with an equal chance. We expect that half of all coin flips will land heads, and half of all coin flips will land tails.

Run the following cell to see the simulation of 10 flips of a fair coin. Let the first item of `coin_proportions` be the proportion of heads and the second item of `coin_proportions` be the proportion of tails.

*Observe what happens when you run this cell multiple times. The proportion of coin flips that land heads and tails appears to change as you are simulating flipping 10 coins each time!*

In [ ]:
coin_proportions = make_array(0.5, 0.5) 
ten_flips = sample_proportions(10, coin_proportions)
ten_flips

`sample_proportions` returns an array that is the same length as the proportion array that is passed through. It contains the proportion of each category that appears in the sample. 

In our example, the first item of `ten_flips` is the simulated proportion of heads and the second item of `ten_flips` is the simulated proportion of tails.

In [ ]:
simluated_proportion_heads = ten_flips.item(0)
simluated_proportion_tails = ten_flips.item(1)

print("In our simluation, " + str(simluated_proportion_heads) + " of flips were heads and " \
      + str(simluated_proportion_tails) + " of flips were tails.")

---

#### Task 10 📍

To begin simulating, we should start by creating a representation of Emily's model to use for our simulation. This will be an array with two items in it. The first item should be the proportion of times, assuming that Emily’s model was correct, a TT practictioner picks the correct hand. The second item should be the proportion of times, under the same assumption, that the TT practitioner picks the incorrect hand. Assign `model_proportions` to this array. 

After this, we can simulate 210 hand choices, as Emily evaluated in real life, and find a single statistic to summarize this instance of the simulation. Use the `sample_proportions` function and assign the proportion of correct hand choices (out of 210) to `simulation_proportion_correct`. Lastly, use your statistic function to assign `one_statistic`  to the value of the statistic for this one simulation.

*Hint:* `sample_proportions` usage can be found [here](http://data8.org/su19/python-reference.html).


In [ ]:
# This saves the random state of our code so that we can 
# generate the same numbers each time we run the code.
# Please do not change this next line. 
np.random.seed(1234)

# Your work goes below this comment.
model_proportions = ...
simulation_proportion_correct = ...
one_statistic = ...
one_statistic

In [ ]:
grader.check("task_10")

---

#### Task 11 📍

Let's now see what the distribution of statistics is actually like under Emily's model. 

1. Define the function `simulation_and_statistic` to take in the `model_proportions` array and the expected proportion of times a TT practitioner would guess a hand correctly under Emily's model. The function should simulate Emily running through the experiment 210 times and return the statistic of this one simulation. _This should follow the same pattern as the code you did in the previous problem._
2. Using this function, assign `simulated_statistics` to an array of 1000 statistics that you calculated under the assumption that Emily's model was true.

In [ ]:
def simulation_and_statistic(model_proportions, expected_proportion_correct):
    '''Simulates 210 TT hand choices under Emily’s model. 
    Returns one statistic from the simulation.'''
    ...

In [ ]:
num_repetitions = 1000

simulated_statistics = ...

for ... in ...:
    ...


In [ ]:
grader.check("task_11")

---

Let's view the distribution of the simulated statistics under Emily's model, and visually compare where the observed statistic lies relative to the simulated statistics.

In [ ]:
t = Table().with_column('Simulated Statistics', simulated_statistics)
t.hist()
plt.scatter(observed_statistic, 0, color='red', s=30, zorder=3);

We can make a visual argument as to whether we believe the observed statistic is consistent with Emily’s model. Here, since larger values of the test statistic suggest the alternative model (where the chance of guessing the correct hand is something other than 50%), we can formalize our analysis by finding what proportion of simulated statistics were as large or larger than our observed test statistic (the area at or to the right of the observed test statistic). If this area is small enough, we’ll declare that the observed data are inconsistent with our simulated model.

---

#### Task 12 📍

Calculate the proportion of simulated statistics greater than or equal to the observed statistic. 

You might want to use `np.count_nonzero` for this.


In [ ]:
proportion_greater_or_equal = ...
proportion_greater_or_equal

In [ ]:
grader.check("task_12")

By convention, we often compare the proportion we just calculated to 0.05. If the proportion of simulated statistics greater than or equal to the observed statistic is sufficiently small (less than or equal to 0.05), then this is evidence against Emily's model. Otherwise, we don’t have any reason to doubt Emily’s model. 

This should help you make your own conclusions about Emily Rosa's experiment. 

Therapeutic touch fell out of use after this experiment, which was eventually accepted into one of the premier medical journals. TT practitioners hit back and accused Emily and her family of tampering with the results, while some claimed that Emily's bad spiritual mood towards therapeutic touch made it difficult to read her HEF. Whatever it may be, Emily's experiment is a classic example about how anyone, with the right resources, can test anything they want!


---

#### Task 13 📍

Now, take some time to reflect on the bulleted question below and then assign `task_13_answer` to an integer representing the best choice from options 1 and 2 below. Choose the best answer.

Question: 
- Is the data more consistent with Emily's model (practioners were randomly guessing)?

1. Since the observed statistic fits within the majority of the histogram data, it seems that the data is not consistent with Emily's model.
2. Since the observed statistic fits within the majority of the histogram data, it seems that the data is consistent with Emily's model.

In [ ]:
task_13_answer = ...

In [ ]:
grader.check("task_13")

## Final Task 📍

<!-- BEGIN QUESTION -->

You're almost done! Run the cell below to check that you passed all of the non-hidden auto-graded tasks, and then follow the instructions below to submit your assignment to Canvas.

In [ ]:
grader.check_all()

<!-- END QUESTION -->

---

## Submit Your Assignment to Canvas

Follow these steps to submit your lab assignment:

1. **Check the Assignment Completion Requirements:** This assignment is scored as Complete or Incomplete. Make sure to check with your instructor about their requirements for a Complete score. 
2. **Run the Auto-Grader:** Ensure you have executed the code cell containing the command `grader.check_all()` above, to run all tests for auto-graded tasks marked with 📍. This command will execute all auto-grader tests sequentially.
3. **Complete Manually Graded Tasks:** Verify that you have responded to all the manually graded tasks marked with 📍🔎.
4. **Save Your Work:** In the notebook's Toolbar, go to `File -> Save Notebook` to save your work and create a checkpoint.
5. **Download the Notebook:** In the notebook's Toolbar, go to `File -> Download HTML` to download the HTML version (`.html`) of this notebook.
6. **Upload to Canvas:** On the Canvas Assignment page, click "Start Assignment" or "New Attempt" to upload the downloaded `.html` file.

---

## Attribution

This content is licensed under the <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0)</a> and derived from the <a href="https://www.data8.org/">Data 8: The Foundations of Data Science</a> offered by the University of California, Berkeley.

<img src="./by-nc-sa.png" width=100px>